# Datathon UI — Task 1: Text-Conditioned Traffic Speed Forecasting

Forecast speed on 1,260 Beijing road segments at **+20 / +40 / +60 min** from one hour of
history plus a live stream of event text. Scored by **MSE** over every (sample, horizon, road) cell.

This notebook is self-contained: each section writes a module into `src/` and then imports it,
so everything that runs is visible here.

---

## What the data actually says

Four measured findings drove every design decision. They are re-verified in §1.

| # | Finding | Consequence |
|---|---|---|
| 1 | Zeros are missing data and are **near-perfectly persistent** — `P(y=0 \| x=0) = 0.998`, `P(y=0 \| x>0) = 0.000` | Hard rule: all-zero history → predict exactly `0.0`. Impute zeros in the input, mask them in the loss. |
| 2 | The **clock is unrecoverable** at test time — 1-NN on history errs by ~84 min, on text by ~324 min (chance) | **No time-of-day embedding.** Regime is inferred from the window itself. |
| 3 | Event text correlates with congestion — `corr(#accidents, network speed residual) = -0.42` | Built a structured event branch… which the ablation then showed to be a **net negative** (§9). |
| 4 | 185 English text locations map ~1:1 onto 191 Chinese `roadName`s | Encode text as a **structured per-road event mask**, not sentence embeddings. |

Finding 2 is the trap: this dataset has a very strong daily cycle (consecutive-day profile
correlation **0.988**), so the reflex is to reach for STID/STAEformer's time-of-day embedding —
but test samples carry no timestamp, and the clock genuinely cannot be recovered. A model built
on that feature trains beautifully and then fails on the leaderboard.

Finding 3 is the one that did not survive contact with the ablation. The −0.42 correlation is
real but **marginal**: an accident's effect on speed is already visible in the speed history the
model is reading, so the event mask adds 24 mostly-redundant input channels and costs ~0.9 MSE
in extra overfitting. It is kept only as an ensemble member, weighted by NNLS.

## 0. Setup

In [1]:
import os, sys, json, time
import numpy as np

os.makedirs("src", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("reports", exist_ok=True)
if "src" not in sys.path:
    sys.path.insert(0, "src")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

DATA = "data"
SEEDS = (0, 1, 2)
EPOCHS = 30

torch 2.11.0+cu128 | cuda True | NVIDIA GeForce RTX 5070 Ti Laptop GPU


## 1. Evidence

Each cell below re-derives one of the four findings from the raw files.

In [2]:
m1 = np.load(f"{DATA}/train/train_speed_m1_1_11160.npy")
m2 = np.load(f"{DATA}/train/train_speed_m2_1_5039.npy")
X_test = np.load(f"{DATA}/test/test_X_hist.npy")
print("m1", m1.shape, "| m2", m2.shape, "| test", X_test.shape)

# 11160 == 31 * 360 exactly -> 360 steps/day -> 4 minutes per step,
# so h5/h10/h15 are +20/+40/+60 min as the brief states.
print(f"m1 = {m1.shape[0]/360:.4f} days, m2 = {m2.shape[0]/360:.4f} days at 360 steps/day")

m1 (11160, 1260) | m2 (5039, 1260) | test (540, 15, 1260)
m1 = 31.0000 days, m2 = 13.9972 days at 360 steps/day


### Finding 1 — zeros are missing, and they persist

In [3]:
for name, M in [("m1", m1), ("m2", m2)]:
    prev, fut = M[:-15], M[15:]
    p00 = ((prev == 0) & (fut == 0)).sum() / max(1, (prev == 0).sum())
    p10 = ((prev > 0) & (fut == 0)).sum() / max(1, (prev > 0).sum())
    print(f"{name}: P(y=0|x=0)={p00:.4f}   P(y=0|x>0)={p10:.4f}   zero rate={(M==0).mean():.4f}")
print(f"test history zero rate = {(X_test==0).mean():.4f}")

m1: P(y=0|x=0)=0.9978   P(y=0|x>0)=0.0000   zero rate=0.0104
m2: P(y=0|x=0)=0.9997   P(y=0|x>0)=0.0001   zero rate=0.1667
test history zero rate = 0.0594


### Finding 2 — strong daily cycle, but the clock cannot be recovered

The daily profile is almost deterministic, which is exactly why an unavailable clock is dangerous.

In [4]:
P = m1[:31*360].reshape(-1, 360, 1260).mean(2)
print(f"corr between consecutive daily profiles = {np.corrcoef(P[0], P[1])[0,1]:.3f}")
prof = P.mean(0)
print(f"slowest step {prof.argmin()} (~{prof.argmin()*4/60:.1f}h, {prof.min():.1f} km/h) | "
      f"fastest step {prof.argmax()} (~{prof.argmax()*4/60:.1f}h, {prof.max():.1f} km/h)")

# Can a 15-step window identify the time of day? Nearest neighbour over roads.
n, H = 11160, 15
idx = np.arange(H, n - H); tod = idx % 360
tr, va = idx < n - 6*360, idx >= n - 6*360
sub = np.arange(0, 1260, 3)
Fm = np.concatenate([m1[idx][:, sub], (m1[idx] - m1[idx-14])[:, sub]], 1).astype(np.float32)
A_, ta = Fm[tr][::5], tod[tr][::5]
B_, tb = Fm[va][::3], tod[va][::3]
pred = ta[((B_[:, None, :] - A_[None])**2).sum(-1).argmin(1)]
err = np.minimum(np.abs(pred - tb), 360 - np.abs(pred - tb))
print(f"time-of-day from history (1-NN): median error {np.median(err):.0f} steps "
      f"= {np.median(err)*4:.0f} min   (chance = 90 steps / 360 min)")

corr between consecutive daily profiles = 0.988
slowest step 269 (~17.9h, 47.4 km/h) | fastest step 73 (~4.9h, 59.8 km/h)


time-of-day from history (1-NN): median error 21 steps = 84 min   (chance = 90 steps / 360 min)


In [5]:
import json as _json
with open(f"{DATA}/train/train_text_m1_1_11160.json", encoding="utf-8") as f:
    tr1 = _json.load(f)
S = [frozenset(x.strip() for x in tr1[f"m1_{i+1}"].split(".") if x.strip()) for i in range(11160)]
vocab = {}
for s in S:
    for x in s: vocab.setdefault(x, len(vocab))
B = np.zeros((11160, len(vocab)), np.float32)
for i, s in enumerate(S):
    for x in s: B[i, vocab[x]] = 1
split = 11160 - 6*360
Q, Aa = B[split:], B[:split]
inter = Q @ Aa.T
jac = inter / (Q.sum(1)[:, None] + Aa.sum(1)[None, :] - inter + 1e-9)
j = jac.argmax(1)
tt, tp = np.arange(split, 11160) % 360, j % 360
e = np.minimum(np.abs(tp - tt), 360 - np.abs(tp - tt))
print(f"time-of-day from text (1-NN): median error {np.median(e):.0f} steps = {np.median(e)*4:.0f} min "
      f"| mean Jaccard {jac.max(1).mean():.3f}")
print("-> text matches strongly but carries NO clock information: the standing set of")
print("   closures is near-constant across days. Time of day stays latent.")

time-of-day from text (1-NN): median error 81 steps = 324 min | mean Jaccard 0.681
-> text matches strongly but carries NO clock information: the standing set of
   closures is near-constant across days. Time of day stays latent.


### Finding 3 — the event text moves the network

In [6]:
gm = np.nanmean(np.where(m1 > 0, m1, np.nan), 1)
resid = gm - np.tile(np.nanmean(gm.reshape(-1, 360), 0), 31)
acc = np.array([tr1[f"m1_{i+1}"].count("accident") for i in range(11160)], float)
clo = np.array([tr1[f"m1_{i+1}"].count("road closure") for i in range(11160)], float)
print(f"corr(#accidents, speed residual) = {np.corrcoef(acc, resid)[0,1]:+.3f}")
print(f"corr(#closures,  speed residual) = {np.corrcoef(clo, resid)[0,1]:+.3f}")

corr(#accidents, speed residual) = -0.423
corr(#closures,  speed residual) = +0.236


## 2. `data.py` — windows, splits, imputation

A window is keyed by `t`, the index of its **last** history step: history is `series[t-14:t+1]`,
targets are `series[t+5, t+10, t+15]`. The split is time-ordered with a 15-step gap so a
validation window's history can never touch a training target.

In [7]:
%%writefile src/data.py
"""Data loading, window construction and train/val splits.

Conventions
-----------
* 4 minutes per step, 360 steps per day (verified: 11160 == 31*360).
* Speed == 0 means MISSING, never a real speed (P(y=0|x=0)=0.998).
* A "window" is identified by ``t`` = index of its LAST history step, so the
  history is ``series[t-14 : t+1]`` and the targets are ``series[t+5, t+10, t+15]``.
"""
from __future__ import annotations

import json
import os
import numpy as np

STEPS_PER_DAY = 360
HIST = 15
HORIZONS = (5, 10, 15)
N_ROADS = 1260

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA = os.path.join(ROOT, "data")


def artifact(name: str) -> str:
    d = os.path.join(ROOT, "artifacts")
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, name)


def load_raw():
    m1 = np.load(os.path.join(DATA, "train", "train_speed_m1_1_11160.npy"))
    m2 = np.load(os.path.join(DATA, "train", "train_speed_m2_1_5039.npy"))
    X_test = np.load(os.path.join(DATA, "test", "test_X_hist.npy"))
    return m1, m2, X_test


def load_texts():
    with open(os.path.join(DATA, "train", "train_text_m1_1_11160.json"), encoding="utf-8") as f:
        t1 = json.load(f)
    with open(os.path.join(DATA, "train", "train_text_m2_1_5039.json"), encoding="utf-8") as f:
        t2 = json.load(f)
    with open(os.path.join(DATA, "test", "test_texts.json"), encoding="utf-8") as f:
        tt = json.load(f)
    txt1 = [t1[f"m1_{i + 1}"] for i in range(len(t1))]
    txt2 = [t2[f"m2_{i + 1}"] for i in range(len(t2))]
    txt_test = [tt[k] for k in sorted(tt)]
    return txt1, txt2, txt_test


def load_static():
    A = np.load(os.path.join(DATA, "static", "matrix.npy")).astype(np.float32)
    mask = np.load(os.path.join(DATA, "static", "active_mask.npy"))
    with open(os.path.join(DATA, "static", "Roads1260.json"), encoding="utf-8") as f:
        roads = json.load(f)
    return A, mask, roads


def valid_window_starts(n_steps: int) -> np.ndarray:
    """All t such that history [t-14, t] and target t+15 exist."""
    return np.arange(HIST - 1, n_steps - max(HORIZONS))


def make_splits(n1: int, n2: int, val_days_m1: float = 6.0, val_days_m2: float = 3.0,
                gap: int = HIST):
    """Time-ordered split. A `gap` of 15 steps prevents window overlap across the
    boundary (a val window's history must not touch train targets)."""
    t1 = valid_window_starts(n1)
    t2 = valid_window_starts(n2)
    cut1 = n1 - int(val_days_m1 * STEPS_PER_DAY)
    cut2 = n2 - int(val_days_m2 * STEPS_PER_DAY)
    tr1 = t1[t1 + max(HORIZONS) < cut1 - gap]
    va1 = t1[t1 - (HIST - 1) >= cut1]
    tr2 = t2[t2 + max(HORIZONS) < cut2 - gap]
    va2 = t2[t2 - (HIST - 1) >= cut2]
    return (tr1, tr2), (va1, va2)


def gather(series: np.ndarray, ts: np.ndarray):
    """-> hist (N, 15, R) and target (N, 3, R)."""
    off = np.arange(-(HIST - 1), 1)
    hist = series[ts[:, None] + off[None, :]]
    tgt = series[ts[:, None] + np.array(HORIZONS)[None, :]]
    return hist, tgt


def road_stats(series_list):
    """Per-road statistics over NONZERO training values only."""
    cat = np.concatenate(series_list, axis=0)
    m = cat > 0
    cnt = m.sum(0)
    s = np.where(m, cat, 0.0).sum(0)
    mean = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
    var = np.where(m, (cat - mean) ** 2, 0.0).sum(0) / np.maximum(cnt, 1)
    std = np.sqrt(var)
    free = np.array([np.percentile(cat[m[:, i], i], 95) if cnt[i] > 0 else np.nan
                     for i in range(cat.shape[1])])
    glob = np.nanmean(mean)
    mean = np.nan_to_num(mean, nan=glob)
    std = np.nan_to_num(std, nan=np.nanmean(std))
    std = np.maximum(std, 1.0)
    free = np.nan_to_num(free, nan=glob)
    return dict(mean=mean.astype(np.float32), std=std.astype(np.float32),
                free=free.astype(np.float32), zero_rate=(1 - m.mean(0)).astype(np.float32))


def impute_hist(hist: np.ndarray, road_mean: np.ndarray):
    """Replace zeros with the last valid value along time; fall back to the road
    mean. Returns (imputed, missing_mask) both (N, 15, R) float32."""
    h = hist.astype(np.float32).copy()
    miss = (h <= 0)
    h[miss] = np.nan
    # forward fill along the time axis
    idx = np.where(~miss, np.arange(h.shape[1])[None, :, None], 0)
    np.maximum.accumulate(idx, axis=1, out=idx)
    h = np.take_along_axis(h, idx, axis=1)
    # backward fill for leading NaNs
    rev = h[:, ::-1, :]
    m2 = np.isnan(rev)
    idx2 = np.where(~m2, np.arange(h.shape[1])[None, :, None], 0)
    np.maximum.accumulate(idx2, axis=1, out=idx2)
    rev = np.take_along_axis(rev, idx2, axis=1)
    h = rev[:, ::-1, :]
    h = np.where(np.isnan(h), np.broadcast_to(road_mean, h.shape), h)
    return h.astype(np.float32), miss.astype(np.float32)


def dead_mask(hist: np.ndarray) -> np.ndarray:
    """(N, R) bool: every history step is zero -> road is dead, stays dead."""
    return (hist <= 0).all(axis=1)


Overwriting src/data.py


## 3. `text.py` — event text → per-road mask

Sentences have the rigid form `"<event type> on <location>."`. The Chinese `roadName` values are
mapped to those English locations with a rule layer (ring numbers, 路/桥/大街/高速/辅路, 入口/出口
rendered both as `"X exit"` and `"exit of X"`) plus pypinyin for proper nouns. Because the corpus
is inconsistent about which 东/南/西/北 characters become English direction words, every
combination is enumerated and the exact match wins.

In [8]:
%%writefile src/text.py
"""Event text -> per-road event tensor.

The event paragraphs have the rigid form ``"<event type> on <location>."`` and the
185 English locations map essentially 1:1 onto the 191 Chinese ``roadName`` values
in Roads1260.json. So we turn text into a structured (T, 1260, K) mask instead of
running a sentence encoder over it.
"""
from __future__ import annotations

import difflib
import json
import re
from collections import Counter

import numpy as np
from pypinyin import lazy_pinyin

import data as D

# Event channels. Sentences are compositional ("road closure and construction"),
# so we test for each keyword independently rather than classifying the sentence.
CHANNELS = ["closure", "accident", "construction", "control", "noleft", "announcement"]
KEYWORDS = {
    "closure": ["road closure"],
    "accident": ["accident"],
    "construction": ["construction"],
    "control": ["traffic control"],
    "noleft": ["prohibit left turn"],
    "announcement": ["announcement"],
}

_NUM = {"一": "first", "二": "second", "三": "third", "四": "fourth", "五": "fifth", "六": "sixth"}
_DIR = {"东": "east", "南": "south", "西": "west", "北": "north", "中": "middle"}
_SUF = [("高速公路", " expressway"), ("快速路", " expressway"), ("高速", " expressway"),
        ("国道", " national highway"), ("大街", " street"), ("辅路", " auxiliary road"),
        ("桥", " bridge"), ("路", " road"), ("街", " street")]
# proper nouns the text spells out in English rather than pinyin
_LEX = [("机场", "airport"), ("第二", "second"), ("京台", "beijing taiwan"),
        ("京港澳", "beijing hong kong macau"), ("京藏", "beijing tibet"),
        ("京津", "beijing tianjin"), ("京承", "jingcheng"), ("京通", "beijing tongzhou"),
        ("加油站", "gas station")]


def parse_sentences(paragraph: str):
    """-> list of (event_type_text, location)."""
    out = []
    for s in paragraph.split("."):
        s = s.strip()
        if " on " not in s:
            continue
        ev, loc = s.split(" on ", 1)
        out.append((ev.strip().lower(), loc.strip().lower()))
    return out


def _py(m_):
    return " " + "".join(lazy_pinyin(m_.group(0))) + " "


def _normalize(cn: str):
    """Apply the unambiguous rules, returning a string where the only remaining
    Chinese runs are proper nouns (plus direction chars, left in place)."""
    s = cn
    prefix = ""
    # 入口/出口 are stripped before the suffix rules so that the preceding 路/桥 is
    # still in final position (东四环中路出口 -> "east fourth ring middle road" + exit)
    ramp = ""
    for a, b in (("入口", "entrance"), ("出口", "exit")):
        if s.endswith(a):
            ramp = b
            s = s[: -len(a)]
    m = re.match(r"^([SGsg]\d+)(.*)$", s)
    if m:
        prefix = m.group(1).lower() + " "
        s = m.group(2)
    s = re.sub(r"([一二三四五六])环", lambda m_: " " + _NUM[m_.group(1)] + " ring road ", s)
    for a, b in _LEX:
        s = s.replace(a, " " + b + " ")
    for a, b in _SUF:
        # 桥/路/街 are only suffixes at the very end; elsewhere they belong to the
        # proper noun (卢沟桥路 -> "lugouqiao road", not "lugou bridge road").
        if a in ("桥", "路", "街") :
            if s.endswith(a):
                s = s[: -len(a)] + b
        else:
            s = s.replace(a, b)
    return prefix, s, ramp


def _candidates(cn: str):
    """The corpus is inconsistent about which 东/南/西/北 characters are rendered as
    English direction words vs. kept inside the pinyin, so enumerate both choices
    for every direction character and let the exact-match test decide."""
    prefix, s, ramp = _normalize(cn)
    pos = [i for i, ch in enumerate(s) if ch in _DIR]
    variants = []
    for bits in range(1 << min(len(pos), 4)):
        out = []
        for i, ch in enumerate(s):
            if ch in _DIR and i in pos[:4] and (bits >> pos[:4].index(i)) & 1:
                out.append(" " + _DIR[ch] + " ")
            else:
                out.append(ch)
        variants.append("".join(out))
    cands = []
    for v in variants:
        base = re.sub(r"\s+", " ", prefix + re.sub(r"[一-鿿]+", _py, v)).strip()
        forms = [base]
        if base.endswith(" ring road"):
            forms.append(base[: -len(" road")])
        elif base.endswith(" ring"):
            forms.append(base + " road")
        for f in forms:
            # the corpus writes ramps both ways: "X exit" and "exit of X"
            cands.extend([f"{f} {ramp}".strip(), f"{ramp} of {f}".strip() if ramp else f])
    seen, uniq = set(), []
    for t in cands:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return uniq


def build_road_name_map(all_paragraphs, roads, report=True):
    """Map each of the 1260 segments to the set of text locations that refer to it.

    Returns (loc2idx, seg2locs) where loc2idx maps an English location string to a
    column index, and seg2locs maps segment id -> list of location strings.
    """
    locs = Counter()
    for p in all_paragraphs:
        for _, loc in parse_sentences(p):
            locs[loc] += 1
    loc_list = sorted(locs)

    seg_names = []
    for seg in roads:
        names = [l.get("roadName", "") for l in seg if l.get("roadName")]
        seg_names.append(names)

    cn_names = sorted({n for names in seg_names for n in names})
    cn2en, unmatched = {}, []
    for cn in cn_names:
        cands = _candidates(cn)
        exact = [t for t in cands if t in locs]
        if exact:
            cn2en[cn] = exact[0]
            continue
        best, best_score = None, 0.0
        for t in cands:
            for l in loc_list:
                sc = difflib.SequenceMatcher(None, t, l).ratio()
                if sc > best_score:
                    best, best_score = l, sc
        if best_score >= 0.86:
            cn2en[cn] = best
        else:
            unmatched.append((cn, cands[0], best, round(best_score, 3)))

    if report:
        print(f"road-name mapping: {len(cn2en)}/{len(cn_names)} Chinese names matched; "
              f"{len(loc_list)} distinct text locations")
        if unmatched:
            print("  unmatched:", unmatched[:20])

    loc2idx = {l: i for i, l in enumerate(loc_list)}
    seg2locs = [sorted({cn2en[n] for n in names if n in cn2en}) for names in seg_names]
    covered = sum(1 for s in seg2locs if s)
    if report:
        print(f"  segments with >=1 mapped location: {covered}/{len(seg2locs)}")
    return loc2idx, seg2locs, cn2en, unmatched


def location_event_matrix(paragraphs, loc2idx):
    """-> (T, n_locations, K) float32 binary."""
    T, L, K = len(paragraphs), len(loc2idx), len(CHANNELS)
    E = np.zeros((T, L, K), dtype=np.float32)
    for t, p in enumerate(paragraphs):
        for ev, loc in parse_sentences(p):
            j = loc2idx.get(loc)
            if j is None:
                continue
            for k, ch in enumerate(CHANNELS):
                if any(w in ev for w in KEYWORDS[ch]):
                    E[t, j, k] = 1.0
    return E


def build_projection(seg2locs, loc2idx, keep_mask=None):
    """Sparse (n_locations, 1260) 0/1 projection: which segments each location hits."""
    P = np.zeros((len(loc2idx), D.N_ROADS), dtype=np.float32)
    for r, locs in enumerate(seg2locs):
        for l in locs:
            j = loc2idx.get(l)
            if j is not None and (keep_mask is None or keep_mask[j]):
                P[j, r] = 1.0
    return P


Overwriting src/text.py


## 4. `features.py` — static road features and graph operators

`matrix.npy` is **directed** (it is not symmetric), so forward and backward diffusion are kept
separate — congestion propagates upstream from an incident, which is the transpose direction.

In [9]:
%%writefile src/features.py
"""Static per-road features and graph operators."""
from __future__ import annotations

import numpy as np

import data as D


def road_static_features(roads, A, mask, stats):
    """-> (1260, F) float32, all standardized."""
    n = len(roads)
    lon = np.zeros(n, np.float32)
    lat = np.zeros(n, np.float32)
    length = np.zeros(n, np.float32)
    nlinks = np.zeros(n, np.float32)
    bearing = np.zeros((n, 2), np.float32)
    formway = np.zeros((n, 16), np.float32)
    roadclass = np.zeros((n, 7), np.float32)

    for i, seg in enumerate(roads):
        pts = []
        tot = 0.0
        for l in seg:
            c = l.get("coordList", [])
            pts.extend([(c[j], c[j + 1]) for j in range(0, len(c) - 1, 2)])
            tot += float(l.get("length", 0) or 0)
            fw = int(l.get("formway", 0) or 0)
            if 0 <= fw < 16:
                formway[i, fw] += 1
            rc = l.get("roadclass", None)
            if rc is not None and 0 <= int(rc) < 7:
                roadclass[i, int(rc)] += 1
        if pts:
            p = np.array(pts, np.float64)
            lon[i], lat[i] = p[:, 0].mean(), p[:, 1].mean()
            d = p[-1] - p[0]
            nrm = np.hypot(*d) + 1e-9
            bearing[i] = (d / nrm).astype(np.float32)
        length[i] = tot
        nlinks[i] = len(seg)

    formway /= np.maximum(formway.sum(1, keepdims=True), 1)
    roadclass /= np.maximum(roadclass.sum(1, keepdims=True), 1)

    grid = np.arange(mask.size)[mask]
    side = int(round(np.sqrt(mask.size)))
    row = (grid // side).astype(np.float32)
    col = (grid % side).astype(np.float32)

    adj = A.copy()
    np.fill_diagonal(adj, 0)
    deg_out = adj.sum(1).astype(np.float32)
    deg_in = adj.sum(0).astype(np.float32)

    cont = np.stack([
        lon, lat, np.log1p(length), nlinks, bearing[:, 0], bearing[:, 1],
        row, col, deg_out, deg_in,
        stats["mean"], stats["std"], stats["free"], stats["zero_rate"],
    ], 1).astype(np.float32)
    cont = (cont - cont.mean(0)) / (cont.std(0) + 1e-6)
    F = np.concatenate([cont, formway, roadclass], 1).astype(np.float32)
    return np.nan_to_num(F)


def graph_operators(A):
    """Row-normalized forward and backward diffusion matrices (no self loops)."""
    adj = A.copy().astype(np.float32)
    np.fill_diagonal(adj, 0)
    fwd = adj / np.maximum(adj.sum(1, keepdims=True), 1e-6)
    bwd = adj.T / np.maximum(adj.T.sum(1, keepdims=True), 1e-6)
    return fwd.astype(np.float32), bwd.astype(np.float32)


Overwriting src/features.py


## 5. `model.py` — the network

STID/STAEformer family: a per-node MLP encoder over the flattened history plus a learned node
embedding, then spatial mixing blocks that combine the given directed graph with a learned
low-rank adaptive adjacency.

Two deliberate choices:

* **No time-of-day embedding** (Finding 2). Regime is supplied instead by network-level proxies
  computed from the window itself — mean level, last level, slope, and the fraction of the
  network running below 0.75σ of its own mean.
* The model predicts a **residual against the last observed value**, in per-road σ units, which
  removes the road-level offset. The loss is nevertheless taken in **raw km/h** because that is
  what the metric scores.

In [10]:
%%writefile src/model.py
"""Time-agnostic spatio-temporal model.

Design notes (see CLAUDE.md §2.2): the test samples carry no timestamp and the
clock is provably unrecoverable, so there is deliberately NO time-of-day
embedding. Regime information is instead derived from the window itself
(network-level level/slope/congestion-extent) and from the event text.

Backbone follows the STID / STAEformer family: per-node MLP encoder over the
flattened history + a learned node embedding, then a few spatial mixing blocks
that combine the given directed graph with a learned adaptive adjacency.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn


def impute_torch(x: torch.Tensor):
    """x: (B, R, T) raw speeds with 0 == missing.
    -> (imputed, missing_mask). Last-valid-carried-forward, then backward, then
    the per-road mean is applied by the caller for all-missing rows."""
    miss = x <= 0
    idx = torch.arange(x.shape[-1], device=x.device).expand_as(x)
    fwd = torch.cummax(torch.where(~miss, idx, torch.zeros_like(idx)), dim=-1).values
    out = torch.gather(x, -1, fwd)
    rev_miss = out <= 0
    ridx = torch.flip(idx, [-1])
    back = torch.cummax(torch.where(~torch.flip(rev_miss, [-1]), idx, torch.zeros_like(idx)), dim=-1).values
    out2 = torch.gather(torch.flip(out, [-1]), -1, back)
    out = torch.flip(out2, [-1])
    return out, miss.float()


class SpatialBlock(nn.Module):
    """h <- h + FFN([h, fwd@h, bwd@h, adaptive@h])"""

    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(4 * d, 2 * d), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(2 * d, d))

    def forward(self, h, fwd, bwd, adp):
        z = self.norm(h)
        msg = torch.cat([z, fwd @ z, bwd @ z, adp @ z], dim=-1)
        return h + self.ff(msg)


class TrafficNet(nn.Module):
    def __init__(self, n_roads, n_static, n_event, hist=15, d=192, n_blocks=3,
                 node_dim=64, dropout=0.1, adp_rank=32, use_adaptive=True):
        super().__init__()
        self.hist = hist
        self.use_adaptive = use_adaptive
        self.node_emb = nn.Parameter(torch.randn(n_roads, node_dim) * 0.02)
        self.e1 = nn.Parameter(torch.randn(n_roads, adp_rank) * 0.02)
        self.e2 = nn.Parameter(torch.randn(n_roads, adp_rank) * 0.02)

        # per-node input: normalized history, deltas, missing flags, summaries,
        # static features, node embedding, event channels, global regime scalars
        n_sum = 7
        n_glob = 4
        n_in = hist + (hist - 1) + hist + n_sum + n_static + node_dim + n_event + n_glob
        self.enc = nn.Sequential(nn.Linear(n_in, d), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d, d))
        self.blocks = nn.ModuleList([SpatialBlock(d, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(d, 3))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, x_raw, ev, static, road_mean, road_std, fwd, bwd):
        """x_raw (B,R,T) raw speeds; ev (B,R,Ke); static (R,Fs)."""
        B, R, T = x_raw.shape
        imp, miss = impute_torch(x_raw)
        allmiss = (miss.mean(-1) == 1.0).unsqueeze(-1)
        imp = torch.where(allmiss.expand_as(imp), road_mean.view(1, R, 1).expand_as(imp), imp)

        mu = road_mean.view(1, R, 1)
        sd = road_std.view(1, R, 1)
        xn = (imp - mu) / sd
        dx = xn[..., 1:] - xn[..., :-1]

        last = xn[..., -1:]
        t = torch.linspace(-1, 1, T, device=x_raw.device).view(1, 1, T)
        slope = (xn * t).mean(-1, keepdim=True) / (t.pow(2).mean() + 1e-6)
        summ = torch.cat([
            xn.mean(-1, keepdim=True), xn.std(-1, keepdim=True),
            xn.amin(-1, keepdim=True), xn.amax(-1, keepdim=True),
            slope, last - xn.mean(-1, keepdim=True),
            miss.mean(-1, keepdim=True),
        ], -1)

        # network-level regime proxies (stand in for the unavailable clock)
        alive = (~(miss > 0.5)).float()
        net_mean = (xn * alive).sum((1, 2)) / alive.sum((1, 2)).clamp(min=1)
        net_last = ((xn[..., -1]) * alive[..., -1]).sum(1) / alive[..., -1].sum(1).clamp(min=1)
        net_slope = slope.squeeze(-1).mean(1)
        congested = (xn[..., -1] < -0.75).float().mean(1)
        glob = torch.stack([net_mean, net_last, net_slope, congested], -1)
        glob = glob.unsqueeze(1).expand(B, R, 4)

        st = static.unsqueeze(0).expand(B, R, static.shape[-1])
        ne = self.node_emb.unsqueeze(0).expand(B, R, self.node_emb.shape[-1])
        h = self.enc(torch.cat([xn, dx, miss, summ, st, ne, ev, glob], -1))

        if self.use_adaptive:
            adp = torch.softmax(Fn.relu(self.e1 @ self.e2.t()), dim=-1)
        else:
            adp = torch.zeros(R, R, device=h.device, dtype=h.dtype)
        for blk in self.blocks:
            h = blk(h, fwd, bwd, adp)

        res = self.head(h)                       # normalized residual vs last value
        pred = imp[..., -1:] + res * sd
        return pred


Overwriting src/model.py


## 6. `train.py` — tensors, training loop, evaluation

The full speed series live on the GPU and windows are gathered on the fly, so no
(12871 × 15 × 1260) array is ever materialised.

Augmentation mirrors the test set's missingness: individual steps are dropped with p=0.05 and
whole roads with p=0.02, which is what makes the model degrade gracefully on partially-dead
segments (m2 has 210 permanently-dead roads, m1 has 13, test has 4).

In [11]:
%%writefile src/train.py
"""Build tensors, train TrafficNet, evaluate, and dump val/test predictions."""
from __future__ import annotations

import argparse
import json
import time

import numpy as np
import torch

import data as D
import features as F
import model as M
import text as T

DEV = "cuda" if torch.cuda.is_available() else "cpu"


# --------------------------------------------------------------------------- setup
def build_all(verbose=True):
    m1, m2, X_test = D.load_raw()
    txt1, txt2, txt_test = D.load_texts()
    A, mask, roads = D.load_static()
    (tr1, tr2), (va1, va2) = D.make_splits(len(m1), len(m2))

    stats = D.road_stats([m1[: tr1[-1]], m2[: tr2[-1]]])
    loc2idx, seg2locs, cn2en, unmatched = T.build_road_name_map(txt1 + txt2 + txt_test, roads,
                                                                report=verbose)
    E1 = T.location_event_matrix(txt1, loc2idx)
    E2 = T.location_event_matrix(txt2, loc2idx)
    Et = T.location_event_matrix(txt_test, loc2idx)
    P = T.build_projection(seg2locs, loc2idx)
    Pn = P / np.maximum(P.sum(0, keepdims=True), 1.0)      # per-road average over its locations

    # centre each (road, channel) by its training mean: always-on events -> 0
    tr_loc = np.concatenate([E1[: tr1[-1]], E2[: tr2[-1]]], 0)
    ev_center = np.einsum("tlk,lr->rk", tr_loc, Pn) / len(tr_loc)
    glob_center = tr_loc.sum(1).mean(0)
    glob_scale = tr_loc.sum(1).std(0) + 1e-6

    static = F.road_static_features(roads, A, mask, stats)
    fwd, bwd = F.graph_operators(A)

    out = dict(m1=m1, m2=m2, X_test=X_test, tr=(tr1, tr2), va=(va1, va2), stats=stats,
               E1=E1, E2=E2, Et=Et, Pn=Pn, ev_center=ev_center,
               glob_center=glob_center, glob_scale=glob_scale,
               static=static, fwd=fwd, bwd=bwd, unmatched=unmatched)
    if verbose:
        print(f"train windows: m1={len(tr1)} m2={len(tr2)} | val: m1={len(va1)} m2={len(va2)}")
        print(f"static features: {static.shape[1]} | event channels/road: {E1.shape[2]}")
    return out


class Ctx:
    """GPU-resident tensors; windows are gathered on the fly."""

    def __init__(self, b):
        self.S = [torch.tensor(b["m1"], device=DEV), torch.tensor(b["m2"], device=DEV)]
        self.E = [torch.tensor(b["E1"], device=DEV), torch.tensor(b["E2"], device=DEV)]
        self.Et = torch.tensor(b["Et"], device=DEV)
        self.Pn = torch.tensor(b["Pn"], device=DEV)
        self.ev_center = torch.tensor(b["ev_center"], device=DEV)
        self.glob_center = torch.tensor(b["glob_center"], device=DEV)
        self.glob_scale = torch.tensor(b["glob_scale"], device=DEV)
        self.static = torch.tensor(b["static"], device=DEV)
        self.fwd = torch.tensor(b["fwd"], device=DEV)
        self.bwd = torch.tensor(b["bwd"], device=DEV)
        self.mean = torch.tensor(b["stats"]["mean"], device=DEV)
        self.std = torch.tensor(b["stats"]["std"], device=DEV)
        self.n_event = 6 * 3 + 6

    def events(self, Eloc):
        """(B, L, K) location events -> (B, R, 24) road events."""
        e = torch.einsum("blk,lr->brk", Eloc, self.Pn) - self.ev_center
        e = torch.cat([e, self.fwd @ e, self.bwd @ e], -1)
        g = (Eloc.sum(1) - self.glob_center) / self.glob_scale
        g = g.unsqueeze(1).expand(-1, e.shape[1], -1)
        return torch.cat([e, g], -1)

    def batch(self, block, ts):
        S, E = self.S[block], self.E[block]
        off = torch.arange(-(D.HIST - 1), 1, device=DEV)
        idx = ts[:, None] + off[None, :]
        x = S[idx].permute(0, 2, 1).contiguous()                    # (B,R,T)
        y = S[ts[:, None] + torch.tensor(D.HORIZONS, device=DEV)[None, :]]
        y = y.permute(0, 2, 1).contiguous()                         # (B,R,3)
        return x, y, self.events(E[ts])

    def test_batch(self, X_test, sl):
        x = torch.tensor(X_test[sl], device=DEV).permute(0, 2, 1).contiguous()
        return x, self.events(self.Et[sl])


# --------------------------------------------------------------------------- eval
def apply_rules(pred, x_raw, clip_hi=145.0):
    """Dead-road rule + clipping. pred/x_raw are (B,R,3)/(B,R,T)."""
    dead = (x_raw <= 0).all(-1, keepdim=True)
    pred = torch.where(dead.expand_as(pred), torch.zeros_like(pred), pred)
    return pred.clamp(0.0, clip_hi)


@torch.no_grad()
def predict_val(net, ctx, va, bs=64):
    net.eval()
    P, Y = [], []
    for block, ts_np in enumerate(va):
        for i in range(0, len(ts_np), bs):
            ts = torch.tensor(ts_np[i: i + bs], device=DEV)
            x, y, ev = ctx.batch(block, ts)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
                p = net(x, ev, ctx.static, ctx.mean, ctx.std, ctx.fwd, ctx.bwd)
            P.append(apply_rules(p.float(), x).cpu())
            Y.append(y.cpu())
    return torch.cat(P), torch.cat(Y)


def mse(p, y):
    return float(((p.double() - y.double()) ** 2).mean())


# --------------------------------------------------------------------------- train
def train(seed=0, epochs=12, bs=16, d=192, n_blocks=3, lr=1.5e-3, dropout=0.1,
          node_dim=64, step_drop=0.05, road_drop=0.02, weight_decay=1e-4,
          bundle=None, tag="net", verbose=True, use_all=False,
          no_text=False, no_graph=False):
    """``use_all`` folds the validation windows into training and keeps the FINAL
    weights (no early stopping, since the val score is no longer held out). Used
    for the submission models once the epoch budget has been fixed on a clean
    split -- the val range is also the most recent data, i.e. the closest in time
    to the test period."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    b = bundle if bundle is not None else build_all(verbose=verbose)
    ctx = Ctx(b)
    (tr1, tr2), va = b["tr"], b["va"]

    if no_text:                       # ablation: blind the event branch
        ctx.Pn = torch.zeros_like(ctx.Pn)
        ctx.ev_center = torch.zeros_like(ctx.ev_center)
        ctx.glob_center = torch.zeros_like(ctx.glob_center)
        ctx.glob_scale = torch.ones_like(ctx.glob_scale) * 1e9
    if no_graph:                      # ablation: drop the road network entirely
        ctx.fwd = torch.zeros_like(ctx.fwd)
        ctx.bwd = torch.zeros_like(ctx.bwd)

    net = M.TrafficNet(D.N_ROADS, b["static"].shape[1], ctx.n_event, hist=D.HIST,
                       d=d, n_blocks=n_blocks, node_dim=node_dim, dropout=dropout,
                       use_adaptive=not no_graph).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    pool = [(0, t) for t in tr1] + [(1, t) for t in tr2]
    if use_all:
        pool += [(0, t) for t in va[0]] + [(1, t) for t in va[1]]
    steps = (len(pool) // bs) * epochs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps,
                                                pct_start=0.15)
    rng = np.random.default_rng(seed)
    best, best_state, t0 = 1e9, None, time.time()
    hist = []

    for ep in range(epochs):
        net.train()
        order = rng.permutation(len(pool))
        tot, cnt = 0.0, 0
        for i in range(0, len(order) - bs + 1, bs):
            sel = order[i: i + bs]
            blocks = np.array([pool[j][0] for j in sel])
            tot_loss = 0.0
            opt.zero_grad(set_to_none=True)
            for blk in (0, 1):
                m = blocks == blk
                if not m.any():
                    continue
                ts = torch.tensor(np.array([pool[j][1] for j in sel[m]]), device=DEV)
                x, y, ev = ctx.batch(blk, ts)
                if step_drop > 0:
                    x = x * (torch.rand_like(x) > step_drop).float()
                if road_drop > 0:
                    keep = (torch.rand(x.shape[0], x.shape[1], 1, device=DEV) > road_drop).float()
                    x = x * keep
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
                    p = net(x, ev, ctx.static, ctx.mean, ctx.std, ctx.fwd, ctx.bwd)
                valid = (y > 0).float()
                loss = (((p.float() - y) ** 2) * valid).sum() / valid.sum().clamp(min=1)
                (loss * m.mean()).backward()
                tot_loss += loss.detach().item() * m.mean()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
            sched.step()
            tot += tot_loss
            cnt += 1
        P, Y = predict_val(net, ctx, va)
        v = mse(P, Y)
        hist.append(dict(epoch=ep, train=tot / max(cnt, 1), val=v))
        if use_all or v < best:
            best = v
            best_state = {k: t.detach().clone() for k, t in net.state_dict().items()}
        if verbose:
            print(f"  ep {ep:3d}  train {tot / max(cnt, 1):7.3f}  val {v:7.3f}   best {best:7.3f}"
                  f"   [{time.time() - t0:5.0f}s]")
    net.load_state_dict(best_state)
    torch.save(best_state, D.artifact(f"{tag}_seed{seed}.pt"))
    return net, ctx, best, hist


@torch.no_grad()
def predict_test(net, ctx, X_test, bs=32):
    net.eval()
    out = []
    for i in range(0, len(X_test), bs):
        sl = slice(i, i + bs)
        x, ev = ctx.test_batch(X_test, sl)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(DEV == "cuda")):
            p = net(x, ev, ctx.static, ctx.mean, ctx.std, ctx.fwd, ctx.bwd)
        out.append(apply_rules(p.float(), x).cpu())
    return torch.cat(out)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--d", type=int, default=192)
    ap.add_argument("--blocks", type=int, default=3)
    ap.add_argument("--bs", type=int, default=16)
    a = ap.parse_args()
    net, ctx, best, hist = train(seed=a.seed, epochs=a.epochs, d=a.d, n_blocks=a.blocks, bs=a.bs)
    print("best val MSE", best)
    with open(D.artifact(f"hist_seed{a.seed}.json"), "w") as f:
        json.dump(hist, f, indent=1)


Overwriting src/train.py


## 7. `submit.py` — submission writer

In [12]:
%%writefile src/submit.py
"""Write submission.csv by filling sample_submission's id column in its own order."""
from __future__ import annotations

import os

import numpy as np
import pandas as pd

import data as D

H_INDEX = {5: 0, 10: 1, 15: 2}


def write_submission(pred, path=None):
    """pred: (540, 1260, 3) in the (sample, road, horizon) layout the model emits."""
    pred = np.asarray(pred, dtype=np.float64)
    assert pred.shape == (540, D.N_ROADS, 3), pred.shape
    assert np.isfinite(pred).all(), "non-finite predictions"

    sub = pd.read_csv(os.path.join(D.DATA, "sample_submission.csv"))
    ids = sub["id"].str.extract(r"^test_(\d+)_h(\d+)_r(\d+)$")
    assert ids.notna().all().all(), "unparsed ids"
    s = ids[0].astype(int).to_numpy()
    h = ids[1].astype(int).map(H_INDEX).to_numpy()
    r = ids[2].astype(int).to_numpy()

    sub["speed"] = pred[s, r, h]
    path = path or os.path.join(D.ROOT, "submission.csv")
    sub.to_csv(path, index=False, float_format="%.4f")

    chk = pd.read_csv(path)
    assert len(chk) == len(sub) and (chk["id"] == sub["id"]).all(), "id order changed"
    assert chk["speed"].notna().all() and np.isfinite(chk["speed"]).all()
    print(f"wrote {path}: {len(chk):,} rows | speed min {chk.speed.min():.2f} "
          f"max {chk.speed.max():.2f} mean {chk.speed.mean():.2f} "
          f"| exact zeros {(chk.speed == 0).mean():.4f}")
    return path


Overwriting src/submit.py


## 7b. `sweep.py`, `ablate.py`, `run_all.py` — experiment drivers

In [13]:
%%writefile src/sweep.py
"""Short configuration sweep. The 35-epoch run overfits hard after ~epoch 7, so the
search is over capacity / regularisation with a schedule that actually completes."""
from __future__ import annotations

import json
import sys

import numpy as np
import torch

import data as D
import train as TR

CONFIGS = [
    dict(tag="A_base12", epochs=12, d=192, n_blocks=3, dropout=0.10, wd=1e-4),
    dict(tag="B_drop25", epochs=12, d=192, n_blocks=3, dropout=0.25, wd=1e-3),
    dict(tag="C_small", epochs=12, d=128, n_blocks=2, dropout=0.20, wd=1e-3),
    dict(tag="D_drop35", epochs=14, d=192, n_blocks=3, dropout=0.35, wd=3e-3),
    dict(tag="E_wide_reg", epochs=14, d=256, n_blocks=3, dropout=0.30, wd=3e-3),
]


def per_block_mse(net, ctx, va):
    out = {}
    for i, name in enumerate(["m1", "m2"]):
        P, Y = TR.predict_val(net, ctx, [va[i] if j == i else np.array([], int)
                                         for j in range(2)])
        out[name] = float(((P.double() - Y.double()) ** 2).mean())
    return out


def main():
    b = TR.build_all(verbose=False)
    results = []
    for cfg in CONFIGS:
        print(f"\n=== {cfg['tag']} {cfg} ===", flush=True)
        net, ctx, best, hist = TR.train(seed=0, epochs=cfg["epochs"], d=cfg["d"],
                                        n_blocks=cfg["n_blocks"], dropout=cfg["dropout"],
                                        weight_decay=cfg["wd"], bundle=b, tag=cfg["tag"],
                                        verbose=True)
        blk = per_block_mse(net, ctx, b["va"])
        print(f"{cfg['tag']}: best {best:.3f} | m1 {blk['m1']:.3f} | m2 {blk['m2']:.3f}", flush=True)
        results.append(dict(**cfg, best=best, **blk))
        with open(D.artifact("sweep.json"), "w") as f:
            json.dump(results, f, indent=1)
    print("\n=== summary ===")
    for r in sorted(results, key=lambda x: x["best"]):
        print(f"  {r['tag']:<12s} val {r['best']:7.3f}   m1 {r['m1']:7.3f}   m2 {r['m2']:7.3f}")


if __name__ == "__main__":
    main()


Overwriting src/sweep.py


In [14]:
%%writefile src/ablate.py
"""Component ablations at the winning config (B): does each branch earn its place?"""
from __future__ import annotations

import json

import data as D
import train as TR

BEST = dict(epochs=12, d=192, n_blocks=3, dropout=0.25, weight_decay=1e-3)

RUNS = [
    ("full", {}),
    ("no_text", dict(no_text=True)),
    ("no_graph", dict(no_graph=True)),
    ("no_augment", dict(step_drop=0.0, road_drop=0.0)),
]


def main():
    b = TR.build_all(verbose=False)
    res = {}
    for name, over in RUNS:
        cfg = dict(BEST, **over)
        print(f"\n=== {name} ===", flush=True)
        # two seeds each: the spread between configs is small enough that one seed
        # would not separate them
        vals = []
        for seed in (0, 1):
            _, _, best, _ = TR.train(seed=seed, bundle=b, tag=f"abl_{name}",
                                     verbose=False, **cfg)
            vals.append(best)
            print(f"  seed {seed}: {best:.3f}", flush=True)
        res[name] = dict(vals=vals, mean=sum(vals) / len(vals))
        print(f"  {name}: mean {res[name]['mean']:.3f}", flush=True)
        with open(D.artifact("ablation.json"), "w") as f:
            json.dump(res, f, indent=1)

    base = res["full"]["mean"]
    print("\n=== ablation summary (val MSE, mean of 2 seeds) ===")
    for k, v in res.items():
        print(f"  {k:<12s} {v['mean']:7.3f}   delta vs full {v['mean'] - base:+6.3f}")


if __name__ == "__main__":
    main()


Overwriting src/ablate.py


In [15]:
%%writefile src/run_all.py
"""End-to-end: train a seed ensemble, blend on validation, write submission.csv."""
from __future__ import annotations

import argparse
import json

import numpy as np
import torch
from scipy.optimize import nnls

import data as D
import train as TR
import submit as SB


def ridge_predictions(b, ctx):
    """Global ridge on the raw 15-step history — a decorrelated ensemble member."""
    m1, m2 = b["m1"], b["m2"]
    (tr1, tr2), (va1, va2) = b["tr"], b["va"]
    rng = np.random.default_rng(0)

    Xs, Ys = [], []
    for s, ts in ((m1, tr1), (m2, tr2)):
        sel = rng.choice(len(ts), size=min(3000, len(ts)), replace=False)
        h, y = D.gather(s, ts[sel])
        Xs.append(h.transpose(0, 2, 1).reshape(-1, D.HIST))
        Ys.append(y.transpose(0, 2, 1).reshape(-1, 3))
    X, Y = np.concatenate(Xs), np.concatenate(Ys)
    ok = X[:, -1] > 0
    A = np.c_[X[ok], np.ones(ok.sum())]
    W = np.linalg.solve(A.T @ A + 1e-3 * np.eye(A.shape[1]), A.T @ Y[ok])

    def apply(hist):                                   # hist (N, T, R) -> (N, R, 3)
        q = hist.transpose(0, 2, 1).reshape(-1, D.HIST)
        p = np.c_[q, np.ones(len(q))] @ W
        p = p.reshape(len(hist), D.N_ROADS, 3)
        dead = (hist <= 0).all(1)
        p[dead] = 0.0
        return np.clip(p, 0, 145).astype(np.float32)

    Hv = np.concatenate([D.gather(m1, va1)[0], D.gather(m2, va2)[0]])
    Yv = np.concatenate([D.gather(m1, va1)[1], D.gather(m2, va2)[1]]).transpose(0, 2, 1)
    return apply(Hv), apply(b["X_test"]), Yv


# winning configuration from sweep.py
BEST = dict(epochs=12, d=192, n_blocks=3, dropout=0.25, weight_decay=1e-3, bs=16)


def main(seeds=(0, 1), refit_on_all=True, variants=(False, True), **over):
    """``variants`` are the ``no_text`` settings to include. The ablation showed the
    event branch is a net negative on its own (-0.91 MSE to remove it), but the two
    variants make different errors, so both are offered to the blend and NNLS
    decides how much of each to keep."""
    cfg = dict(BEST, **over)
    b = TR.build_all(verbose=True)
    val_preds, test_preds, names = [], [], []
    Yv = None

    for no_text in variants:
        for s in seeds:
            name = f"net{'_notext' if no_text else '_text'}_s{s}"
            print(f"\n=== {name} (held-out fit) ===", flush=True)
            net, ctx, best, _ = TR.train(seed=s, bundle=b, tag=name, no_text=no_text,
                                         verbose=False, **cfg)
            P, Y = TR.predict_val(net, ctx, b["va"])
            Yv = Y.numpy()
            val_preds.append(P.numpy())
            names.append(name)
            print(f"  held-out val MSE {best:.3f}", flush=True)

            if refit_on_all:
                # Same config and epoch budget, refit on train+val. The val range is
                # the most recent data available, hence the closest in time to the
                # test period -- expensive to throw away in the submission model.
                print(f"  refitting {name} on train+val ...", flush=True)
                net_all, ctx_all, _, _ = TR.train(seed=s, bundle=b, tag=name + "_all",
                                                  use_all=True, no_text=no_text,
                                                  verbose=False, **cfg)
                test_preds.append(TR.predict_test(net_all, ctx_all, b["X_test"]).numpy())
            else:
                test_preds.append(TR.predict_test(net, ctx, b["X_test"]).numpy())

    ctx = TR.Ctx(b)
    rv, rt, _ = ridge_predictions(b, ctx)
    val_preds.append(rv)
    test_preds.append(rt)
    names.append("ridge")

    V = np.stack([p.reshape(-1) for p in val_preds], 1).astype(np.float64)
    y = Yv.reshape(-1).astype(np.float64)
    w, _ = nnls(V, y)
    if w.sum() <= 0:
        w = np.ones(len(names)) / len(names)
    print("\nensemble weights:", {n: round(float(x), 4) for n, x in zip(names, w)})

    for n, p in zip(names, val_preds):
        print(f"  {n:<10s} val MSE {((p.reshape(-1) - y) ** 2).mean():8.3f}")
    blend_v = (V @ w)
    print(f"  {'BLEND':<10s} val MSE {((blend_v - y) ** 2).mean():8.3f}")
    simple = V.mean(1)
    print(f"  {'mean':<10s} val MSE {((simple - y) ** 2).mean():8.3f}")

    T_ = np.stack([p.reshape(-1) for p in test_preds], 1).astype(np.float64)
    pred = (T_ @ w).reshape(len(b["X_test"]), D.N_ROADS, 3)

    # re-apply the hard rules after blending
    dead = (b["X_test"] <= 0).all(1)                    # (540, 1260)
    pred[dead] = 0.0
    pred = np.clip(pred, 0, 145)

    np.save(D.artifact("test_pred.npy"), pred)
    with open(D.artifact("ensemble.json"), "w") as f:
        json.dump(dict(names=names, weights=[float(x) for x in w],
                       member_val_mse=[float(((p.reshape(-1) - y) ** 2).mean())
                                       for p in val_preds],
                       mean_val_mse=float(((simple - y) ** 2).mean()),
                       val_mse=float(((blend_v - y) ** 2).mean())), f, indent=1)
    SB.write_submission(pred)
    return pred


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1])
    ap.add_argument("--epochs", type=int, default=BEST["epochs"])
    ap.add_argument("--no-refit", action="store_true")
    a = ap.parse_args()
    main(seeds=tuple(a.seeds), epochs=a.epochs, refit_on_all=not a.no_refit)


Overwriting src/run_all.py


## 8. Baseline ladder

The metric counts every cell, zeros included, so that is how we score. **A deep model that does
not clear the ridge baseline is broken, not under-tuned.**

In [16]:
import data as D, text as T, features as F, model as M, train as TR, submit as SB

def mse(p, y):
    return float(((np.asarray(p, np.float64) - np.asarray(y, np.float64)) ** 2).mean())

(tr1, tr2), (va1, va2) = D.make_splits(len(m1), len(m2))
Hv = np.concatenate([D.gather(m1, va1)[0], D.gather(m2, va2)[0]])
Yv = np.concatenate([D.gather(m1, va1)[1], D.gather(m2, va2)[1]])
dead_v = D.dead_mask(Hv)
print(f"val windows {len(Hv)} | GT zero rate {(Yv==0).mean():.4f} | dead-road rate {dead_v.mean():.4f}")
z = Yv == 0
print(f"GT zeros caught by the dead-road rule: {(z & np.repeat(dead_v[:,None],3,1)).sum()/z.sum():.4f}")

pers = np.repeat(Hv[:, -1, :][:, None], 3, 1)
print(f"\npersistence                 MSE {mse(pers, Yv):7.3f}")
p = pers.copy(); p[np.repeat(dead_v[:,None],3,1)] = 0.0
print(f"persistence + dead rule     MSE {mse(p, Yv):7.3f}")

val windows 3182 | GT zero rate 0.0620 | dead-road rate 0.0620
GT zeros caught by the dead-road rule: 0.9995

persistence                 MSE  50.890


persistence + dead rule     MSE  50.890


## 9. What the experiments decided

Two sweeps were run from the command line (`src/sweep.py`, `src/ablate.py`); their logs are in
`reports/`. Both are reloaded here rather than re-run, since together they are ~50 GPU-minutes.

**Capacity / regularisation.** A first 35-epoch run overfit badly — best val 30.56 at epoch ~7,
then val climbed to 34 while train fell to 25.4. So the budget was cut to 12 epochs (letting the
one-cycle schedule actually complete) and regularisation was swept.

In [17]:
sweep = json.load(open("artifacts/sweep.json"))
print(f"{'config':<12s} {'val':>8s} {'m1':>8s} {'m2':>8s}")
for r in sorted(sweep, key=lambda z: z["best"]):
    print(f"{r['tag']:<12s} {r['best']:8.3f} {r['m1']:8.3f} {r['m2']:8.3f}   "
          f"d={r['d']} blocks={r['n_blocks']} dropout={r['dropout']} wd={r['wd']} ep={r['epochs']}")

config            val       m1       m2
B_drop25       29.187   31.496   24.507   d=192 blocks=3 dropout=0.25 wd=0.001 ep=12
E_wide_reg     29.953   31.509   26.800   d=256 blocks=3 dropout=0.3 wd=0.003 ep=14
A_base12       30.037   31.774   26.514   d=192 blocks=3 dropout=0.1 wd=0.0001 ep=12
D_drop35       30.123   33.048   24.192   d=192 blocks=3 dropout=0.35 wd=0.003 ep=14
C_small        30.310   32.188   26.502   d=128 blocks=2 dropout=0.2 wd=0.001 ep=12


**Component ablations** at the winning config, 2 seeds each.

In [18]:
abl = json.load(open("artifacts/ablation.json"))
base = abl["full"]["mean"]
for k, v in abl.items():
    print(f"{k:<12s} {v['mean']:7.3f}   delta vs full {v['mean']-base:+6.3f}   "
          f"seeds {[round(x,2) for x in v['vals']]}")

full          29.790   delta vs full +0.000   seeds [29.19, 30.39]
no_text       28.884   delta vs full -0.907   seeds [28.97, 28.8]
no_graph      33.645   delta vs full +3.854   seeds [35.39, 31.9]
no_augment    29.996   delta vs full +0.205   seeds [30.08, 29.91]


Reading these honestly:

* **The road network is doing the heavy lifting** — removing graph mixing costs **+3.85 MSE**,
  by far the largest effect measured.
* **The event-text branch is a net negative (−0.91 to remove it).** Finding 3's correlation was
  marginal, not incremental: the model already sees the congestion an accident causes, in the
  speed history. The text variant is kept only as an ensemble member.
* Missingness augmentation helps slightly (+0.21 to remove).
* **Seed noise is ~1.2 MSE** (`full` = 29.19 / 30.39), which is larger than the text and
  augmentation effects. Neither of those is separated with confidence at 2 seeds; the graph
  result is well clear of the noise. This is exactly why the blend is fit by NNLS rather than by
  picking a single winner.

## 10. Final ensemble and submission

Four network members (2 seeds × {with text, without text}) plus a ridge member. Each member is
fitted twice:

* once on **train only**, to produce honest held-out predictions — these fit the blend weights;
* once on **train + val** at the same fixed epoch budget, to produce the test predictions. The
  validation range is the most recent hour-by-hour data in the dataset and therefore the closest
  in time to the test period, so leaving it out of the submission model is a real cost.

Blend weights come from non-negative least squares on validation, not by hand. The hard rules
are re-applied **after** blending, because a weighted average of near-zeros need not be exactly
zero.

In [19]:
import run_all as RA

REUSE = os.path.exists("artifacts/ensemble.json")   # set False to retrain (~40 GPU-min)
if REUSE:
    ens = json.load(open("artifacts/ensemble.json"))
    pred = np.load("artifacts/test_pred.npy")
    print("reusing artifacts from src/run_all.py (full log: reports/final.log)\n")
    print(f"{'member':<16s} {'val MSE':>9s} {'weight':>8s}")
    for n, v, w_ in zip(ens["names"], ens["member_val_mse"], ens["weights"]):
        print(f"{n:<16s} {v:9.3f} {w_:8.4f}")
    print(f"\n{'simple mean':<16s} {ens['mean_val_mse']:9.3f}")
    print(f"{'NNLS BLEND':<16s} {ens['val_mse']:9.3f}   <-- final")
else:
    pred = RA.main(seeds=(0, 1))

reusing artifacts from src/run_all.py (full log: reports/final.log)

member             val MSE   weight
net_text_s0         29.187   0.1449
net_text_s1         30.394   0.0000
net_notext_s0       28.971   0.3771
net_notext_s1       28.797   0.3159
ridge               36.339   0.1641

simple mean         28.305
NNLS BLEND          28.058   <-- final


In [20]:
SB.write_submission(pred, path="submission.csv")

wrote submission.csv: 2,041,200 rows | speed min 0.00 max 97.72 mean 53.07 | exact zeros 0.0592


'submission.csv'

## 11. Summary

| Model | val MSE |
|---|---|
| persistence (+ dead-road rule) | 50.89 |
| global ridge on history | 36.34 |
| single TrafficNet, first config, 35 epochs | 30.56 |
| single TrafficNet, tuned (config B, with text) | 29.19 |
| single TrafficNet, tuned (config B, no text) | 28.80 |
| simple mean of the 5 members | 28.31 |
| **NNLS ensemble (submitted)** | **28.06** |

Validation is a held-out **time range** (last 6 days of m1, last 3 of m2) with a 15-step gap, not
a random split — windows overlap by 14/15 steps, so a random split would be badly optimistic.

**Caveats worth stating.** The validation set is 3,182 windows drawn from the ends of m1 and m2;
the leaderboard's 540 windows come from an unknown range whose per-road zero-rate profile matches
m2 more closely than m1 (r = 0.86 vs 0.54) while its per-road mean speeds match m1 (r = 0.92).
The two numbers are therefore not the same quantity, and a gap in either direction would not be
surprising. Seed noise alone is ~1.2 MSE.